In [ ]:
!pip install -q transformers accelerate sentence-transformers chromadb pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import pandas as pd
import chromadb
import torch

from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
def load_text_file(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return f.read()

In [ ]:
from pypdf import PdfReader

n_path = "/content/drive/MyDrive/data_handling_and_machine_learning/RAG/NFLX/10-K_2026-01-23_0001065280-26-000034.txt"
c_path = "/content/drive/MyDrive/data_handling_and_machine_learning/RAG/KO/10-K_2026-02-20_0001628280-26-010047.txt"

n_reader = load_text_file(n_path)
c_reader = load_text_file(c_path)

print("Netflix characters:", len(n_reader))
print("Coca-Cola characters:", len(c_reader))

print("Netflix words:", len(n_reader.split()))
print("Coca-Cola words:", len(c_reader.split()))

Netflix characters: 309861
Coca-Cola characters: 661405
Netflix words: 43418
Coca-Cola words: 96120


In [ ]:
documents = []

for path, company in [
    (n_path, "Netflix"),
    (c_path, "Coca-Cola")
]:
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append({
        "company": company,
        "text": text
    })

print(documents[0]["company"])
print(documents[0]["text"][:1000])

print(documents[1]["company"])
print(documents[1]["text"][:1000])

Netflix
nflx-20251231
0001065280
2025
FY
false
434
428
426
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
http://fasb.org/us-gaap/2025#Liabilities
http://fasb.org/us-gaap/2025#Liabilities
P1Y
http://fasb.org/us-gaap/2025#OtherPensionPlansPostretirementOrSupplementalPlansDefinedBenefitMember
iso4217:USD
xbrli:shares
iso4217:USD
xbrli:shares
xbrli:pure
nflx:option
iso4217:EUR
nflx:vote
nflx:segment
0001065280
2025-01-01
2025-12-31
0001065280
2025-06-30
0001065280
2025-12-31
0001065280
2025-10-01
2025-12-31
0001065280
nflx:SpencerNeumannMember
2025-10-01
2025-12-31
0001065280
nflx:SpencerNeumannMember
nflx:GregPetersPlanAdoptionMember
2025-10-01
2025-12-31
0001065280
nflx:SpencerNeumannMember
2025-12-31
0001065280
nf

In [ ]:
def chunk_text(text, chunk_size=700, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

In [ ]:
netflix_chunks = chunk_text(n_reader)
coca_cola_chunks = chunk_text(c_reader)

print("Netflix chunks:", len(netflix_chunks))
print("Coca-Cola chunks:", len(coca_cola_chunks))

print("\nFirst Netflix chunk:")
print(netflix_chunks[0])

print("\nFirst Coca-Cola chunk:")
print(coca_cola_chunks[0])

Netflix chunks: 517
Coca-Cola chunks: 1103

First Netflix chunk:
nflx-20251231
0001065280
2025
FY
false
434
428
426
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
http://fasb.org/us-gaap/2025#Liabilities
http://fasb.org/us-gaap/2025#Liabilities
P1Y
http://fasb.org/us-gaap/2025#OtherPensionPlansPostretirementOrSupplementalPlansDefinedBenefitMember
iso4217:USD
xbrli:shares
iso4217:USD
xbrli:shares
xbrli:pure
nflx:option
iso4217:EUR
nflx:vote
nflx:segment
0001065280
2025-01-01
2025-12-3

First Coca-Cola chunk:
ko-20251231
0000021344
2025
FY
False
http://fasb.org/us-gaap/2025#AssetsOfDisposalGroupIncludingDiscontinuedOperation http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#AssetsOfDisp

In [ ]:
all_chunks = []

documents = [
    {
        "company": "Netflix",
        "text": n_reader,
        "source": n_path
    },
    {
        "company": "Coca-Cola",
        "text": c_reader,
        "source": c_path
    }
]

for document in documents:

    chunks = chunk_text(
        document["text"],
        chunk_size=700,
        overlap=100
    )

    for chunk_number, chunk in enumerate(chunks):

        all_chunks.append({
            "text": chunk,
            "company": document["company"],
            "chunk": chunk_number,
            "source": document["source"]
        })

print("Total chunks:", len(all_chunks))

Total chunks: 1620


In [ ]:
for item in all_chunks[:3]:
    print("Source:", item["source"])
    print("Company:", item["company"])
    print("Chunk:", item["chunk"])
    print(item["text"][:500])
    print("-" * 80)

Source: /content/drive/MyDrive/data_handling_and_machine_learning/RAG/NFLX/10-K_2026-01-23_0001065280-26-000034.txt
Company: Netflix
Chunk: 0
nflx-20251231
0001065280
2025
FY
false
434
428
426
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
http://fasb.org/us-gaap/2025#Liabilities
http://fasb.org/us-gaap/2025#Liabilities
P1Y
http://fasb.org/us-gaap/2025#OtherPensi
--------------------------------------------------------------------------------
Source: /content/drive/MyDrive/data_handling_and_machine_learning/RAG/NFLX/10-K_2026-01-23_0001065280-26-000034.txt
Company: Netflix
Chunk: 1
rli:shares
xbrli:pure
nflx:option
iso4217:EUR
nflx:vote
nflx:segment
0001065280
2025-01-01
2025-12-31
0001065280
2025-06-30
0001065280

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
texts = [item["text"] for item in all_chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Number of embeddings: 1620
Embedding shape: (1620, 384)


In [ ]:
import chromadb

client = chromadb.PersistentClient(
    path="./chroma_db"
)

In [ ]:
collection = client.get_or_create_collection(
    name="rag_documents"
)

In [ ]:
ids = [
    f"{item['source']}_company_{item['company']}_chunk_{item['chunk']}"
    for item in all_chunks
]

In [ ]:
metadatas = [
    {
        "source": item["source"],
        "company": item["company"],
        "chunk": item["chunk"]
    }
    for item in all_chunks
]

In [ ]:
collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

In [ ]:
print("Documents in collection:", collection.count())

Documents in collection: 1620


In [ ]:
result = collection.get(
    limit=3,
    include=["documents", "metadatas"]
)

for document, metadata in zip(
    result["documents"],
    result["metadatas"]
):
    print(metadata)
    print(document[:300])
    print("-" * 80)

{'company': 'Netflix', 'source': '/content/drive/MyDrive/data_handling_and_machine_learning/RAG/NFLX/10-K_2026-01-23_0001065280-26-000034.txt', 'chunk': 0}
nflx-20251231
0001065280
2025
FY
false
434
428
426
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#OtherLia
--------------------------------------------------------------------------------
{'company': 'Netflix', 'chunk': 1, 'source': '/content/drive/MyDrive/data_handling_and_machine_learning/RAG/NFLX/10-K_2026-01-23_0001065280-26-000034.txt'}
rli:shares
xbrli:pure
nflx:option
iso4217:EUR
nflx:vote
nflx:segment
0001065280
2025-01-01
2025-12-31
0001065280
2025-06-30
0001065280
2025-12-31
0001065280
2025-10-01
2025-12-31
0001065280
nflx:SpencerNeumannMember
2025-10-01
2025-12-31
0001065280
nflx:SpencerNeumannMember
nflx:GregPetersPlanAdopti
-----

In [ ]:
question = "What are the Netflix's main business segments?"

In [ ]:
question2 = "What was Coca-Cola's net income in 2025?"

In [ ]:
question_embedding = embedding_model.encode(
    [question, question2]
)

In [ ]:
results = collection.query(
    query_embeddings=[question_embedding[0].tolist()],
    n_results=4
)

results2 = collection.query(
    query_embeddings=[question_embedding[1].tolist()],
    n_results=4
)

In [ ]:
for i, document in enumerate(results["documents"][0]):

    print(f"Result {i + 1}")
    print("Source:", results["metadatas"][0][i])
    print(document)
    print("-" * 80)

Result 1
Source: {'chunk': 74, 'company': 'Netflix', 'source': '/content/drive/MyDrive/data_handling_and_machine_learning/RAG/NFLX/10-K_2026-01-23_0001065280-26-000034.txt'}
included throughout this filing and particularly in Item 1A:
“
Risk Factors
”
section set forth in this Annual Report on Form 10-K. All forward-looking statements included in this document are based on information available to us on the date hereof, and we assume no obligation to revise or publicly release any revision to any such forward-looking statement, except as may otherwise be required by law.
Item 1.
Business
ABOUT US
Netflix, Inc. (“Netflix”, the “Company”, “registrant”, “we”, or “us”) is one of the world’s leading entertainment services offering TV series, films, games and live programming across a wide variety of genres and languages. Members can play, pause and resume watching a
--------------------------------------------------------------------------------
Result 2
Source: {'source': '/content/drive/M

In [ ]:
for i, document in enumerate(results2["documents"][0]):

    print(f"Result {i + 1}")
    print("Source:", results["metadatas"][0][i])
    print(document)
    print("-" * 80)

Result 1
Source: {'chunk': 74, 'company': 'Netflix', 'source': '/content/drive/MyDrive/data_handling_and_machine_learning/RAG/NFLX/10-K_2026-01-23_0001065280-26-000034.txt'}
VE INCOME
(In millions)
Year Ended December 31,
2025
2024
2023
Consolidated Net Income
$
13,137
$
10,649
$
10,703
Other Comprehensive Income:
Net foreign currency translation adjustments
2,868
(
2,893
)
736
Net gains (losses) on derivatives
(
360
)
270
(
178
)
Net change in unrealized gains (losses) on available-for-sale debt securities
38
(
63
)
24
Net change in pension and other postretirement benefit liabilities
94
109
(
109
)
Total Comprehensive Income
15,777
8,072
11,176
Less: Comprehensive income (loss) attributable to noncontrolling interests
(
42
)
9
(
158
)
Total Comprehensive Income Attributable to Shareowners of
The Coca-Cola Company
$
15,819
$
8,063
$
11,334
Refer to Notes to Con
--------------------------------------------------------------------------------
Result 2
Source: {'source': '/content/drive/

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [ ]:
print(model.device)

cuda:0


In [ ]:
def generate_answer(prompt, max_new_tokens=200):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [ ]:
def build_context(results):

    context_parts = []

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    for document, metadata in zip(documents, metadatas):

        source = metadata["source"]
        company = metadata["company"]

        context_parts.append(
            f"[Source: {source}, Company: {company}]\n"
            f"{document}"
        )

    return "\n\n".join(context_parts)

In [ ]:
def build_rag_prompt(question, context):

    return f"""
You are answering a question using the provided document context.

Rules:
1. Use the provided context as the primary source of information.
2. Do not invent facts that are not supported by the context.
3. If the context does not contain enough information, say:
   "I don't know based on the provided documents."
4. Give a concise and direct answer.

DOCUMENT CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

In [ ]:
def rag_answer(
    question,
    n_results=4,
    max_new_tokens=200
):

    # 1. Embed the question
    question_embedding = embedding_model.encode(
        question
    )

    # 2. Retrieve relevant chunks
    results = collection.query(
        query_embeddings=[
            question_embedding.tolist()
        ],
        n_results=n_results
    )

    # 3. Build context
    context = build_context(results)

    # 4. Build prompt
    prompt = build_rag_prompt(
        question,
        context
    )

    # 5. Generate answer
    answer = generate_answer(
        prompt,
        max_new_tokens=max_new_tokens
    )

    return {
        "question": question,
        "answer": answer,
        "context": context,
        "results": results
    }

In [ ]:
question = "Which company has higher revenue according to the filings?"

result = rag_answer(question)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

QUESTION:
Which company has higher revenue according to the filings?

ANSWER:
Netflix has a higher revenue than Coca-Cola. According to the filings, Netflix reported net operating revenues of $45,183,036 thousand in 2025, while Coca-Cola reported net operating revenues of $47,941 million in 2025. This indicates that Netflix had significantly more revenue than Coca-Cola during this period.


In [ ]:
for i, metadata in enumerate(
    result["results"]["metadatas"][0]
):

    print(
        f"Retrieved source {i + 1}: "
        f"{metadata['source']}, "
        f"company {metadata['company']}, "
        f"chunk {metadata['chunk']}"
    )

Retrieved source 1: /content/drive/MyDrive/data_handling_and_machine_learning/RAG/KO/10-K_2026-02-20_0001628280-26-010047.txt, company Coca-Cola, chunk 978
Retrieved source 2: /content/drive/MyDrive/data_handling_and_machine_learning/RAG/NFLX/10-K_2026-01-23_0001065280-26-000034.txt, company Netflix, chunk 339
Retrieved source 3: /content/drive/MyDrive/data_handling_and_machine_learning/RAG/KO/10-K_2026-02-20_0001628280-26-010047.txt, company Coca-Cola, chunk 627
Retrieved source 4: /content/drive/MyDrive/data_handling_and_machine_learning/RAG/KO/10-K_2026-02-20_0001628280-26-010047.txt, company Coca-Cola, chunk 518


In [ ]:
questions = [
    "What are Coca-Cola's main sources of revenue",
    "Based on the filing, what factors could support Netflix's future growth?",
    "What are the company's main competitive advantages mentioned in the filing?"
]

for question in questions:

    result = rag_answer(question)

    print("=" * 80)
    print("QUESTION:", question)
    print("ANSWER:", result["answer"])

QUESTION: What are Coca-Cola's main sources of revenue
ANSWER: Coca-Cola's main sources of revenue include:

1. Sales of its flagship product, Coca-Cola, across various regions and markets.
2. Revenue generated from the sale of sparkling flavors like Sprite and Fanta.
3. Profits from the sale of water, sports, coffee, and tea beverages.
4. Value-added dairy and plant-based beverage sales.
5. Emerging beverage category revenues. 

These diverse offerings cater to different consumer preferences and geographical locations, ensuring a broad customer base and stable revenue streams. The company also derives additional income through the sale of subsidiary shares and acquisitions of interests held by non-controlling owners. Additionally, it reinvests earnings back into the business, contributing to ongoing growth and expansion efforts.
QUESTION: Based on the filing, what factors could support Netflix's future growth?
ANSWER: Based on the filing, several factors could support Netflix's future

In [ ]:
question = "What is the population of Mars according to this document?"

result = rag_answer(question)

print(result["answer"])

I don't know based on the provided documents. 

This answer is derived because the given text does not provide any specific information about the population of Mars. It only discusses Coca-Cola's financial data and some general economic trends without mentioning Mars or its inhabitants. Therefore, there isn't enough information to determine the population of Mars from the provided context.


In [ ]:
retrieval_values = [1, 2, 4, 6, 8]

In [ ]:
for k in retrieval_values:

    result = rag_answer(
        "What were Netflix's largest expenses?",
        n_results=k
    )

    print("=" * 80)
    print("Retrieved chunks:", k)
    print(result["answer"])

Retrieved chunks: 1
Based on the provided document context, Netflix's largest expense appears to be Stock-based compensation expense, which amounted to $368,449 for the year in question. This expense represents the cost associated with granting stock options or other equity awards to employees and reflects the value of these awards over time. While the context doesn't provide specific details about how this expense compares to others, it stands out as the most significant item among those listed.
Retrieved chunks: 2
Based on the provided document context, Netflix's largest expenses were:

1. Cost of revenues - $23,275,329
2. Sales and marketing - $3,301,306
3. Technology and development - $3,391,390
4. General and administrative - $1,888,408

These four categories accounted for most of Netflix's operating income. However, it's important to note that the total cost of revenues includes both costs associated with producing content and selling products, which can be broken down further in